## base_model--> non_instruction_model--> instruction_model--> preference_model

In [1]:
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments, Trainer
from peft import LoraConfig, get_peft_model, TaskType
from datasets import load_dataset

In [2]:
base_model = "TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T"

In [3]:
tokenizer = AutoTokenizer.from_pretrained(base_model)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/560 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/776 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

In [6]:
import zipfile
import os
# Path to your zip file
zip_path = "/content/tinyllama-pharma.zip"

# Extract all files
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall()

In [7]:
model_path = "/content/content/tinyllama-pharma/checkpoint-5"

In [9]:
!pip install --upgrade torchao>=0.16.0

In [10]:
instruction_model = AutoModelForCausalLM.from_pretrained(model_path, device_map="auto")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/176 [00:00<?, ?it/s]

In [11]:
prompt = "Explain how artificial intelligence is improving the process of drug discovery and development in the pharmaceutical industry."

In [16]:
import os

# Path to the chat template file
chat_template_path = "/content/content/tinyllama-pharma/chat_template.jinja"

# Check if the file exists before trying to open it
if os.path.exists(chat_template_path):
    with open(chat_template_path, "r") as f:
        tokenizer.chat_template = f.read()
else:
    print(f"Warning: Chat template file not found at {chat_template_path}. Please ensure it exists or provide a default template.")

messages = [
    {
        "role": "user",
        "content": "Explain how artificial intelligence is improving the process of drug discovery and development in the pharmaceutical industry."
    }
]

# Convert messages -> prompt text
prompt = tokenizer.apply_chat_template(
    messages,

    tokenize=False,

    # Adds assistant generation token
    add_generation_prompt=True
)


In [17]:
inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

In [18]:
outputs = instruction_model.generate(
    **inputs,
    max_new_tokens=100,
    temperature=0.8,
    top_p=0.9,
    do_sample=True,
    repetition_penalty=1.1
)

In [19]:
print("\nModel Output:\n")
print(tokenizer.decode(outputs[0], skip_special_tokens=True))


Model Output:

<|user|>
Explain how artificial intelligence is improving the process of drug discovery and development in the pharmaceutical industry. 
<|assistant|>
Artificial intelligence (AI) has been gaining significant attention as a tool for improving drug discovery and development in the pharmaceutical industry. Here are some ways AI is contributing to this process:

1. Data-driven approach: AI algorithms can analyze vast amounts of data from various sources such as genetic sequences, molecular structures, drug-like compounds, and patient samples to identify promising candidates that meet specific therapeutic needs


## Now lets start with prefrence base tuning or preference based alignment

In [20]:
!pip install -U trl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 751.0/751.0 kB 17.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 36.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 17.6 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0


In [1]:
!pip install -U bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 13.2 MB/s eta 0:00:00


In [2]:
from trl import DPOTrainer
from transformers import AutoTokenizer,  AutoModelForCausalLM, TrainingArguments
from peft import PeftModel
from peft import LoraConfig, get_peft_model, TaskType
from datasets import load_dataset
import torch

In [3]:
base_model = "TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T"

In [4]:
instruction_checkpoint = "/content/content/tinyllama-pharma/checkpoint-5"

In [5]:
# Load dataset
dataset = load_dataset("csv", data_files="/content/pharma_preference_data.csv")["train"]

Generating train split: 0 examples [00:00, ? examples/s]

In [6]:
tokenizer = AutoTokenizer.from_pretrained(base_model)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [7]:
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

get_peft_model() → Create a new LoRA during training

PeftModel.from_pretrained() → Load an already-trained LoRA for inference or further training

In [8]:
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=["q_proj", "v_proj"],
    bias="none"
)

In [ ]:
# pref_model_lora = get_peft_model(instruction_model, lora_config)

/usr/local/lib/python3.12/dist-packages/peft/mapping_func.py:72: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:282: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


In [9]:
base_model

'TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T'

In [11]:
#STEP A: Load base
from transformers import BitsAndBytesConfig

quantization_config = BitsAndBytesConfig(
    load_in_8bit=True
)

model = AutoModelForCausalLM.from_pretrained(
    base_model,
    quantization_config=quantization_config,
    device_map="auto"
)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/129 [00:00<?, ?B/s]

In [12]:
#STEP B: Load Instruction LoRA + merge
model = PeftModel.from_pretrained(model, instruction_checkpoint)

In [13]:
model = model.merge_and_unload()

/usr/local/lib/python3.12/dist-packages/peft/tuners/lora/bnb.py:97: UserWarning: Merge lora module to 8-bit linear may get different generations due to rounding errors.
  warnings.warn(


In [14]:
#STEP C: Attach NEW LoRA for preference
pref_model_lora = get_peft_model(model, lora_config)

| Stage           | What You Should Do                       | Wrong Way (you did)      |
| --------------- | ---------------------------------------- | ------------------------ |
| Non-Instruction | Base + LoRA                              | ✔ correct                |
| Instruction     | Base + **merge(stage1 LoRA)** + NEW LoRA | ❌ “LoRA on LoRA”         |
| Preference      | Base + **merge(stage2 LoRA)** + NEW LoRA | ❌ “LoRA on LoRA on LoRA” |


In [15]:
import os
os.environ["WANDB_DISABLED"] = "true"

In [16]:
from trl import DPOTrainer, DPOConfig

In [17]:
dpo_args = DPOConfig(
    output_dir="./tinyllama-preference-alignment",
    learning_rate=2e-5,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    num_train_epochs=1,
    beta=0.1,
    report_to=None,
    logging_dir=None, # disable logging to wandb or tensorboard
    loss_type="sigmoid",  # or "hinge", depending on experiment
    remove_unused_columns=False
)


In [19]:
dpo_args.report_to = []

trainer = DPOTrainer(
    model=pref_model_lora,
    ref_model=None,
    args=dpo_args,
    train_dataset=dataset,
    processing_class=tokenizer,   # instead of tokenizer argument
    # you can pass data_collator if needed,
    # optionally eval_dataset etc.
)

In [20]:
trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 2}.
/usr/local/lib/python3.12/dist-packages/bitsandbytes/autograd/_functions.py:123: UserWarning: MatMul8bitLt: inputs will be cast from torch.float32 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")
/usr/local/lib/python3.12/dist-packages/bitsandbytes/autograd/_functions.py:123: UserWarning: MatMul8bitLt: inputs will be cast from torch.bfloat16 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")


Step,Training Loss


TrainOutput(global_step=1, training_loss=0.6931471824645996, metrics={'train_runtime': 5.0141, 'train_samples_per_second': 0.997, 'train_steps_per_second': 0.199, 'total_flos': 4366854955008.0, 'train_loss': 0.6931471824645996})

### Testing with Non-Instruction Model

In [21]:
question = "Explain how Metformin works in the human body and why some researchers believe it could have benefits beyond diabetes treatment."

In [23]:
import zipfile
import os
# Path to your zip file
zip_path = "/content/tinyllama-non-instruction.zip"

# Extract all files
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall()

In [24]:
model_path = "/content/checkpoint-5"
non_instruction_model = AutoModelForCausalLM.from_pretrained(model_path, device_map="auto")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/88 [00:00<?, ?it/s]

In [25]:
inputs = tokenizer(question, return_tensors="pt").to("cuda")

In [26]:
outputs = non_instruction_model.generate(
    **inputs,
    max_new_tokens=100,
    temperature=0.8,
    top_p=0.9,
    do_sample=True,
    repetition_penalty=1.1
)

In [27]:
print("\nModel Output:\n")
print(tokenizer.decode(outputs[0], skip_special_tokens=True))


Model Output:

Explain how Metformin works in the human body and why some researchers believe it could have benefits beyond diabetes treatment.
Diabetic retinopathy is a serious condition that affects millions of people worldwide, with the potential to cause blindness if not treated early. The researchers, led by the University of Cambridge, found that diabetic retinopathy was caused by an imbalance between two types of blood cells known as macrophages. They showed that blocking the enzyme that produces these macrophages helped treat the condition.
Their findings could lead


### Testing with Instruction-Fine-Tuned Model

In [28]:
model_path = "/content/content/tinyllama-pharma/checkpoint-5"
instruction_model = AutoModelForCausalLM.from_pretrained(model_path, device_map="auto")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/176 [00:00<?, ?it/s]

In [32]:
import os

# Path to the chat template file
chat_template_path = "/content/content/tinyllama-pharma/chat_template.jinja"

# Check if the file exists before trying to open it
if os.path.exists(chat_template_path):
    with open(chat_template_path, "r") as f:
        tokenizer.chat_template = f.read()
else:
    print(f"Warning: Chat template file not found at {chat_template_path}. Please ensure it exists or provide a default template.")

messages = [
    {
        "role": "user",
        "content": question
    }
]

# Convert messages -> prompt text
prompt = tokenizer.apply_chat_template(
    messages,

    tokenize=False,

    # Adds assistant generation token
    add_generation_prompt=True
)


In [33]:
inputs = tokenizer(question, return_tensors="pt").to("cuda")

In [34]:
outputs = instruction_model.generate(
    **inputs,
    max_new_tokens=100,
    temperature=0.8,
    top_p=0.9,
    do_sample=True,
    repetition_penalty=1.1
)


In [35]:
print("\nModel Output:\n")
print(tokenizer.decode(outputs[0], skip_special_tokens=True))


Model Output:

Explain how Metformin works in the human body and why some researchers believe it could have benefits beyond diabetes treatment.

Dr. Smith: Sure, Metformin is a type of medicine that works by reducing the production of insulin. Insulin is a hormone that helps regulate blood sugar levels, so when we don't produce enough, our bodies may store more glucose as glycogen (sugar) instead of releasing it into the bloodstream. This can lead to high blood sugar levels and other health problems. By reducing the production of ins


### Testing with DPO (Preference-Aligned) Model

In [36]:
model_path = "/content/tinyllama-preference-alignment/checkpoint-1"

In [37]:
preference_aligned_model = AutoModelForCausalLM.from_pretrained(model_path, dtype=torch.float16)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/88 [00:00<?, ?it/s]

In [38]:
preference_aligned_model.to("cuda")

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(32000, 2048)
    (layers): ModuleList(
      (0-21): 22 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): lora.Linear(
            (base_layer): Linear(in_features=2048, out_features=2048, bias=False)
            (lora_dropout): ModuleDict(
              (default): Dropout(p=0.05, inplace=False)
            )
            (lora_A): ModuleDict(
              (default): Linear(in_features=2048, out_features=8, bias=False)
            )
            (lora_B): ModuleDict(
              (default): Linear(in_features=8, out_features=2048, bias=False)
            )
            (lora_embedding_A): ParameterDict()
            (lora_embedding_B): ParameterDict()
            (lora_magnitude_vector): ModuleDict()
          )
          (k_proj): Linear(in_features=2048, out_features=256, bias=False)
          (v_proj): lora.Linear(
            (base_layer): Linear(in_features=2048, out_features=256, bia

In [39]:
inputs = tokenizer(question, return_tensors="pt").to("cuda")

In [44]:
outputs = preference_aligned_model.generate(
    **inputs,
    max_new_tokens=100,
    temperature=0.2,
    top_p=0.7,
    do_sample=True,
    repetition_penalty=1.1
)

In [45]:
print("\nModel Output:\n")
print(tokenizer.decode(outputs[0], skip_special_tokens=True))


Model Output:

Explain how Metformin works in the human body and why some researchers believe it could have benefits beyond diabetes treatment.
Metformin is a drug that has been used for decades to treat type 2 diabetes, but its use is now being expanded to other conditions as well. It's also known by several other names, including Glucophage, Glucophage XR, and Glucophage ER.
Metformin is an oral medication that works by reducing the amount of glucose (sugar) produced by the liver. This helps reduce
